# Nothing Ever Happens — 95c Convergence Backtest

Theory: buy YES at 95c whenever a market reaches that price. Does it converge to $1?

Data: Kalshi tennis markets, July 13-15 2026. Read-only queries against live DB.

In [ ]:
import sqlite3, pandas as pd, matplotlib.pyplot as plt, numpy as np

DB = '/Users/farquaad/kalshi-ghost-trader/kalshi_tennis.db'
conn = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)

plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})
pd.set_option('display.width', 120)

## 1. Data Overview

In [ ]:
overview = pd.read_sql_query('''
SELECT
  (SELECT COUNT(*) FROM events) AS total_events,
  (SELECT COUNT(*) FROM markets) AS total_markets,
  (SELECT COUNT(DISTINCT market_ticker) FROM ticks) AS markets_with_ticks,
  (SELECT COUNT(*) FROM ticks) AS total_ticks,
  (SELECT COUNT(DISTINCT match_ticker) FROM points) AS matches_with_points,
  (SELECT COUNT(*) FROM points) AS total_points,
  (SELECT datetime(MIN(ts)/1000, 'unixepoch') FROM ticks) AS first_tick,
  (SELECT datetime(MAX(ts)/1000, 'unixepoch') FROM ticks) AS last_tick
''', conn)
overview.T

In [ ]:
result_dist = pd.read_sql_query('''
SELECT result, COUNT(*) AS markets
FROM markets WHERE result IS NOT NULL AND result != ''
GROUP BY result
''', conn)
result_dist.set_index('result').plot(kind='bar', legend=False, title='Market Results (settled)')
plt.xticks(rotation=0)
plt.ylabel('Markets')
plt.tight_layout()
plt.show()

## 2. How Many Markets Touched 95c+?

Filtering out phantom ticks: empty orderbooks show ask=1.0 with size=0. Not tradeable.
Only counting ticks where `yes_ask_size > 0`.

In [ ]:
thresholds = pd.read_sql_query('''
WITH t AS (
  SELECT 0.90 AS thr UNION ALL SELECT 0.92 UNION ALL
  SELECT 0.95 UNION ALL SELECT 0.97 UNION ALL SELECT 0.99
)
SELECT t.thr AS threshold,
  (SELECT COUNT(DISTINCT market_ticker) FROM ticks WHERE yes_ask >= t.thr AND yes_ask_size > 0
   AND market_ticker IN (SELECT market_ticker FROM markets WHERE result IN ('yes','no'))) AS markets
FROM t
''', conn)
thresholds.set_index('threshold').plot(kind='bar', legend=False)
plt.title('Markets Touching Threshold (tradeable, size > 0)')
plt.xlabel('Yes Ask Threshold')
plt.ylabel('Markets')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
thresholds

## 3. Naive Strategy: Buy at 95c Anytime

One entry per market (first 95c touch). Hold to settlement.
P&L per share: +1-entry if YES, -entry if NO.

In [ ]:
naive = pd.read_sql_query('''
WITH entry AS (
  SELECT tk.market_ticker AS mt, tk.yes_ask AS entry_price, tk.ts, m.close_ts, m.result,
    ROW_NUMBER() OVER (PARTITION BY tk.market_ticker ORDER BY tk.ts) AS rn
  FROM ticks tk
  JOIN markets m ON m.market_ticker = tk.market_ticker
  WHERE tk.yes_ask >= 0.95 AND tk.yes_ask_size > 0
    AND m.result IN ('yes','no')
)
SELECT e.mt, e.result, e.entry_price,
  (e.close_ts - e.ts)/1000.0 AS secs_to_close,
  CASE WHEN e.result='yes' THEN (1.0 - e.entry_price) ELSE -e.entry_price END AS pnl_per_share
FROM entry e WHERE e.rn = 1
ORDER BY e.entry_price DESC
''', conn)

total = len(naive)
wins = (naive.result == 'yes').sum()
losses = (naive.result == 'no').sum()
win_pct = 100 * wins / total
net_pnl = naive.pnl_per_share.sum()
print(f'Trades: {total}  Wins: {wins}  Losses: {losses}')
print(f'Win rate: {win_pct:.1f}%  (break-even: 95%)')
print(f'Net P&L: {net_pnl:.2f} per share  ({net_pnl/total:.4f}/share avg)')
print(f'\nBreak-even win rate at avg entry {naive.entry_price.mean():.4f}: '
      f'{naive.entry_price.mean()*100:.1f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['green' if r == 'yes' else 'red' for r in naive.result]
ax1.barh(range(len(naive)), naive.pnl_per_share, color=colors, alpha=0.7)
ax1.set_yticks([])
ax1.set_xlabel('P&L per share')
ax1.set_title(f'Per-trade P&L (naive, anytime)  Net: {net_pnl:.2f}')
ax1.axvline(0, color='black', linewidth=0.8)

ax2.scatter(naive.secs_to_close / 60, naive.entry_price,
            c=colors, alpha=0.7, s=40)
ax2.set_xlabel('Minutes before close')
ax2.set_ylabel('Entry price')
ax2.set_title('Entry timing vs price (green=YES win, red=NO loss)')
ax2.invert_xaxis()
plt.tight_layout()
plt.show()

## 4. Time-Filtered: Within 30 Min of Close

Key insight: the 5 NO losses all occurred 28+ min before close. Near close, convergence is near-certain.

In [ ]:
windows = pd.read_sql_query('''
WITH entry AS (
  SELECT tk.market_ticker AS mt, tk.yes_ask AS entry_price, tk.ts, m.close_ts, m.result,
    ROW_NUMBER() OVER (PARTITION BY tk.market_ticker ORDER BY tk.ts) AS rn
  FROM ticks tk
  JOIN markets m ON m.market_ticker = tk.market_ticker
  WHERE tk.yes_ask >= 0.95 AND tk.yes_ask_size > 0
    AND m.result IN ('yes','no')
)
SELECT window, total, yes_wins, no_losses, yes_pct, pnl_per_share FROM (
  SELECT '<=5min' AS window,
    COUNT(*) AS total, SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END) AS yes_wins,
    SUM(CASE WHEN result='no' THEN 1 ELSE 0 END) AS no_losses,
    ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*), 1) AS yes_pct,
    ROUND(SUM(CASE WHEN result='yes' THEN (1.0-entry_price) ELSE -entry_price END)/COUNT(*), 6) AS pnl_per_share
  FROM entry WHERE rn=1 AND (close_ts-ts) BETWEEN 0 AND 300000
  UNION ALL
  SELECT '5-30min', COUNT(*), SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END),
    SUM(CASE WHEN result='no' THEN 1 ELSE 0 END),
    ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*), 1),
    ROUND(SUM(CASE WHEN result='yes' THEN (1.0-entry_price) ELSE -entry_price END)/COUNT(*), 6)
  FROM entry WHERE rn=1 AND (close_ts-ts) BETWEEN 300001 AND 1800000
  UNION ALL
  SELECT '30min-2hr', COUNT(*), SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END),
    SUM(CASE WHEN result='no' THEN 1 ELSE 0 END),
    ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*), 1),
    ROUND(SUM(CASE WHEN result='yes' THEN (1.0-entry_price) ELSE -entry_price END)/COUNT(*), 6)
  FROM entry WHERE rn=1 AND (close_ts-ts) BETWEEN 1800001 AND 7200000
  UNION ALL
  SELECT '>2hr', COUNT(*), SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END),
    SUM(CASE WHEN result='no' THEN 1 ELSE 0 END),
    ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*), 1),
    ROUND(SUM(CASE WHEN result='yes' THEN (1.0-entry_price) ELSE -entry_price END)/COUNT(*), 6)
  FROM entry WHERE rn=1 AND (close_ts-ts) > 7200000
)
''', conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x = range(len(windows))
ax1.bar(x, windows.yes_pct, color=['#2ca02c','#2ca02c','#d62728','#d62728'], alpha=0.7)
ax1.axhline(95, color='black', linestyle='--', label='Break-even (95%)')
ax1.set_xticks(x); ax1.set_xticklabels(windows.window)
ax1.set_ylabel('Win rate %')
ax1.set_title('Win rate by time-to-close')
ax1.legend()
for i, row in windows.iterrows():
    ax1.text(i, row.yes_pct + 1, f'{row.yes_wins}/{row.total}', ha='center')

colors2 = ['green' if p > 0 else 'red' for p in windows.pnl_per_share]
ax2.bar(x, windows.pnl_per_share, color=colors2, alpha=0.7)
ax2.set_xticks(x); ax2.set_xticklabels(windows.window)
ax2.set_ylabel('P&L per share')
ax2.set_title('P&L per share by time-to-close')
ax2.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()
windows

## 5. Threshold Comparison Near Close (within 30 min)

In [ ]:
thresh_near = pd.read_sql_query('''
WITH e90 AS (
  SELECT tk.market_ticker AS mt, tk.yes_ask AS ep, tk.ts, m.close_ts, m.result,
    ROW_NUMBER() OVER (PARTITION BY tk.market_ticker ORDER BY tk.ts) AS rn
  FROM ticks tk JOIN markets m ON m.market_ticker = tk.market_ticker
  WHERE tk.yes_ask >= 0.90 AND tk.yes_ask_size > 0 AND m.result IN ('yes','no')
),
e95 AS (
  SELECT tk.market_ticker AS mt, tk.yes_ask AS ep, tk.ts, m.close_ts, m.result,
    ROW_NUMBER() OVER (PARTITION BY tk.market_ticker ORDER BY tk.ts) AS rn
  FROM ticks tk JOIN markets m ON m.market_ticker = tk.market_ticker
  WHERE tk.yes_ask >= 0.95 AND tk.yes_ask_size > 0 AND m.result IN ('yes','no')
),
e99 AS (
  SELECT tk.market_ticker AS mt, tk.yes_ask AS ep, tk.ts, m.close_ts, m.result,
    ROW_NUMBER() OVER (PARTITION BY tk.market_ticker ORDER BY tk.ts) AS rn
  FROM ticks tk JOIN markets m ON m.market_ticker = tk.market_ticker
  WHERE tk.yes_ask >= 0.99 AND tk.yes_ask_size > 0 AND m.result IN ('yes','no')
)
SELECT '0.90' AS thr, COUNT(*) AS n, SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END) AS wins,
  SUM(CASE WHEN result='no' THEN 1 ELSE 0 END) AS losses,
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*),1) AS win_pct,
  ROUND(SUM(CASE WHEN result='yes' THEN (1.0-ep) ELSE -ep END)/COUNT(*),6) AS pnl_per_share
FROM e90 WHERE rn=1 AND (close_ts-ts) BETWEEN 0 AND 1800000
UNION ALL
SELECT '0.95', COUNT(*), SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN result='no' THEN 1 ELSE 0 END),
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*),1),
  ROUND(SUM(CASE WHEN result='yes' THEN (1.0-ep) ELSE -ep END)/COUNT(*),6)
FROM e95 WHERE rn=1 AND (close_ts-ts) BETWEEN 0 AND 1800000
UNION ALL
SELECT '0.99', COUNT(*), SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN result='no' THEN 1 ELSE 0 END),
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*),1),
  ROUND(SUM(CASE WHEN result='yes' THEN (1.0-ep) ELSE -ep END)/COUNT(*),6)
FROM e99 WHERE rn=1 AND (close_ts-ts) BETWEEN 0 AND 1800000
''', conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x = range(len(thresh_near))
ax1.bar(x, thresh_near.win_pct, color='#2ca02c', alpha=0.7)
ax1.axhline(95, color='black', linestyle='--', label='Break-even')
ax1.set_xticks(list(x)); ax1.set_xticklabels([f'{t}c' for t in thresh_near.thr])
ax1.set_ylabel('Win rate %'); ax1.set_title('Win rate by threshold (within 30min of close)')
ax1.legend()
for i, row in thresh_near.iterrows():
    ax1.text(i, row.win_pct + 0.5, f'{row.wins}/{row.n}', ha='center')

ax2.bar(x, thresh_near.pnl_per_share * 100, color='#1f77b4', alpha=0.7)
ax2.set_xticks(list(x)); ax2.set_xticklabels([f'{t}c' for t in thresh_near.thr])
ax2.set_ylabel('P&L per share (cents)')
ax2.set_title('P&L per share by threshold')
ax2.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()
thresh_near

## 6. $20 Per Tick at 95c+ (Repeated Buying)

Invest $20 at every tick where ask >= 95c. Shows why repeated buying is dangerous:
losers linger at 95c, sucking in more capital before crashing.

In [ ]:
per_tick = pd.read_sql_query('''
SELECT m.result,
  COUNT(*) AS ticks,
  COUNT(DISTINCT tk.market_ticker) AS markets,
  ROUND(SUM(20.0), 2) AS invested,
  ROUND(SUM(CASE WHEN m.result='yes' THEN 20.0/tk.yes_ask - 20.0 ELSE -20.0 END), 2) AS net_profit
FROM ticks tk
JOIN markets m ON m.market_ticker = tk.market_ticker
WHERE tk.yes_ask >= 0.95 AND tk.yes_ask_size > 0
  AND m.result IN ('yes','no')
GROUP BY m.result
''', conn)

total_invested = per_tick.invested.sum()
total_net = per_tick.net_profit.sum()
print(f'Total invested: ${total_invested:,.2f}')
print(f'Net profit: ${total_net:,.2f}')
print(f'ROI: {100*total_net/total_invested:.2f}%')
print()
print(per_tick.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(per_tick.result, per_tick.invested, color=['#d62728','#2ca02c'], alpha=0.7)
ax1.set_ylabel('Invested ($)'); ax1.set_title('Capital deployed by outcome')
ax2.bar(per_tick.result, per_tick.net_profit, color=['#d62728','#2ca02c'], alpha=0.7)
ax2.set_ylabel('Net P&L ($)'); ax2.set_title('Net P&L by outcome')
ax2.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Per-market breakdown: losers linger at 95c, winners pass through fast
per_market = pd.read_sql_query('''
SELECT tk.market_ticker, m.result,
  COUNT(*) AS ticks_at_95,
  ROUND(SUM(20.0), 2) AS invested,
  ROUND(SUM(CASE WHEN m.result='yes' THEN 20.0/tk.yes_ask - 20.0 ELSE -20.0 END), 2) AS net_profit
FROM ticks tk
JOIN markets m ON m.market_ticker = tk.market_ticker
WHERE tk.yes_ask >= 0.95 AND tk.yes_ask_size > 0
  AND m.result IN ('yes','no')
GROUP BY tk.market_ticker, m.result
ORDER BY net_profit ASC
''', conn)

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['red' if r == 'no' else 'green' for r in per_market.result]
ax.barh(range(len(per_market)), per_market.net_profit, color=colors, alpha=0.7)
ax.set_yticks([])
ax.set_xlabel('Net P&L ($)')
ax.set_title('Per-market P&L ($20/tick at 95c+) — losers drain more capital')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()
print(f'\nTop 5 worst markets:')
print(per_market.head().to_string(index=False))

## 7. Tennis Score Filter: Final Set, Game 8+

Using FlashScore point-by-point data. Buy 95c once when match is in its final set and past game 8.
At that point the upset window has closed — no next set for a comeback.

In [ ]:
score_filter = pd.read_sql_query('''
WITH first_95 AS (
  SELECT tk.market_ticker AS mt, tk.yes_ask AS entry_price, tk.ts, m.event_ticker, m.result, m.close_ts,
    ROW_NUMBER() OVER (PARTITION BY tk.market_ticker ORDER BY tk.ts) AS rn
  FROM ticks tk JOIN markets m ON m.market_ticker = tk.market_ticker
  WHERE tk.yes_ask >= 0.95 AND tk.yes_ask_size > 0 AND m.result IN ('yes','no')
),
match_state AS (
  SELECT f.mt, f.entry_price, f.event_ticker, f.result, f.close_ts, f.ts,
    (SELECT p.set_number FROM points p WHERE p.match_ticker = f.event_ticker AND p.ts_ms <= f.ts ORDER BY p.ts_ms DESC LIMIT 1) AS cs,
    (SELECT p.game_number FROM points p WHERE p.match_ticker = f.event_ticker AND p.ts_ms <= f.ts ORDER BY p.ts_ms DESC LIMIT 1) AS cg,
    (SELECT MAX(p.set_number) FROM points p WHERE p.match_ticker = f.event_ticker) AS ms
  FROM first_95 f WHERE f.rn = 1
)
SELECT 'final set, game 8+' AS filter,
  COUNT(*) AS trades, SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END) AS wins,
  SUM(CASE WHEN result='no' THEN 1 ELSE 0 END) AS losses,
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*),1) AS win_pct,
  20*COUNT(*) AS invested,
  ROUND(SUM(CASE WHEN result='yes' THEN 20.0/entry_price ELSE 0 END),2) AS payout,
  ROUND(SUM(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END),2) AS net_profit,
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END)/(20.0*COUNT(*)),2) AS roi_pct
FROM match_state WHERE cs = ms AND cg >= 8
UNION ALL
SELECT 'final set, game 9+',
  COUNT(*), SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN result='no' THEN 1 ELSE 0 END),
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*),1),
  20*COUNT(*),
  ROUND(SUM(CASE WHEN result='yes' THEN 20.0/entry_price ELSE 0 END),2),
  ROUND(SUM(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END),2),
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END)/(20.0*COUNT(*)),2)
FROM match_state WHERE cs = ms AND cg >= 9
UNION ALL
SELECT 'final set, any game',
  COUNT(*), SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN result='no' THEN 1 ELSE 0 END),
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*),1),
  20*COUNT(*),
  ROUND(SUM(CASE WHEN result='yes' THEN 20.0/entry_price ELSE 0 END),2),
  ROUND(SUM(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END),2),
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END)/(20.0*COUNT(*)),2)
FROM match_state WHERE cs = ms
UNION ALL
SELECT 'set 2-3, game 9+ (not final-set filtered)',
  COUNT(*), SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN result='no' THEN 1 ELSE 0 END),
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 1 ELSE 0 END)/COUNT(*),1),
  20*COUNT(*),
  ROUND(SUM(CASE WHEN result='yes' THEN 20.0/entry_price ELSE 0 END),2),
  ROUND(SUM(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END),2),
  ROUND(100.0*SUM(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END)/(20.0*COUNT(*)),2)
FROM match_state WHERE cs IN (2,3) AND cg >= 9
''', conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
labels = [f.split(',')[0][:18] for f in score_filter['filter']]
x = range(len(score_filter))
colors_w = ['green' if w >= 95 else 'red' for w in score_filter.win_pct]
ax1.bar(x, score_filter.win_pct, color=colors_w, alpha=0.7)
ax1.axhline(95, color='black', linestyle='--', label='Break-even')
ax1.set_xticks(list(x)); ax1.set_xticklabels(labels, rotation=25, ha='right')
ax1.set_ylabel('Win rate %'); ax1.set_title('Win rate by score filter')
ax1.legend()
for i, row in score_filter.iterrows():
    ax1.text(i, row.win_pct + 1, f'{row.wins}/{row.trades}', ha='center', fontsize=8)

colors_p = ['green' if p > 0 else 'red' for p in score_filter.net_profit]
ax2.bar(x, score_filter.net_profit, color=colors_p, alpha=0.7)
ax2.set_xticks(list(x)); ax2.set_xticklabels(labels, rotation=25, ha='right')
ax2.set_ylabel('Net profit ($)')
ax2.set_title('Net profit ($20/trade) by score filter')
ax2.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()
score_filter

In [ ]:
# Detail: individual trades in the best filter (final set, game 8+)
detail = pd.read_sql_query('''
WITH first_95 AS (
  SELECT tk.market_ticker AS mt, tk.yes_ask AS entry_price, tk.ts, m.event_ticker, m.result, m.close_ts,
    ROW_NUMBER() OVER (PARTITION BY tk.market_ticker ORDER BY tk.ts) AS rn
  FROM ticks tk JOIN markets m ON m.market_ticker = tk.market_ticker
  WHERE tk.yes_ask >= 0.95 AND tk.yes_ask_size > 0 AND m.result IN ('yes','no')
),
match_state AS (
  SELECT f.mt, f.entry_price, f.event_ticker, f.result, f.close_ts, f.ts,
    (SELECT p.set_number FROM points p WHERE p.match_ticker = f.event_ticker AND p.ts_ms <= f.ts ORDER BY p.ts_ms DESC LIMIT 1) AS set_no,
    (SELECT p.game_number FROM points p WHERE p.match_ticker = f.event_ticker AND p.ts_ms <= f.ts ORDER BY p.ts_ms DESC LIMIT 1) AS game_no,
    (SELECT MAX(p.set_number) FROM points p WHERE p.match_ticker = f.event_ticker) AS max_set
  FROM first_95 f WHERE f.rn = 1
)
SELECT mt AS market, result, entry_price, set_no, game_no, max_set,
  ROUND((close_ts - ts)/1000.0, 1) AS secs_to_close,
  ROUND(CASE WHEN result='yes' THEN (1.0-entry_price) ELSE -entry_price END, 4) AS pnl_per_share,
  ROUND(CASE WHEN result='yes' THEN 20.0/entry_price - 20.0 ELSE -20.0 END, 2) AS net_on_20
FROM match_state WHERE set_no = max_set AND game_no >= 8
ORDER BY net_on_20 ASC
''', conn)
detail

## 8. Data Completeness Caveat

Only ~62% of settled markets with ticks have price convergence at the end.
Tracker disconnects before settlement for 38% of markets.

In [ ]:
convergence = pd.read_sql_query('''
WITH last_ticks AS (
  SELECT tk.market_ticker, tk.price, tk.yes_ask, tk.yes_bid, tk.ts,
    ROW_NUMBER() OVER (PARTITION BY tk.market_ticker ORDER BY tk.ts DESC) AS rn
  FROM ticks tk
  JOIN markets m ON m.market_ticker = tk.market_ticker
  WHERE m.result IS NOT NULL AND m.result != ''
)
SELECT
  CASE
    WHEN result='yes' AND price >= 0.90 THEN 'yes converged'
    WHEN result='yes' AND price < 0.90 THEN 'yes NOT converged'
    WHEN result='no' AND price <= 0.10 THEN 'no converged'
    WHEN result='no' AND price > 0.10 THEN 'no NOT converged'
    ELSE 'other'
  END AS convergence,
  COUNT(*) AS markets
FROM last_ticks lt
JOIN markets m ON m.market_ticker = lt.market_ticker
WHERE lt.rn = 1
GROUP BY convergence
''', conn)

fig, ax = plt.subplots(figsize=(8, 5))
colors = {'yes converged': '#2ca02c', 'yes NOT converged': '#ff9896',
          'no converged': '#2ca02c', 'no NOT converged': '#ff9896', 'other': '#cccccc'}
ax.bar(convergence.convergence, convergence.markets,
       color=[colors.get(c, '#999') for c in convergence.convergence], alpha=0.7)
ax.set_ylabel('Markets')
ax.set_title('Last-tick price vs result (convergence check)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()
convergence

## 9. Price Trajectories for 95c Markets

Sample of markets that touched 95c, showing how price evolved. Winners shoot to $1, losers crash.

In [ ]:
# Pick 12 markets that touched 95c: 8 winners, 4 losers
sample_markets = pd.read_sql_query('''
WITH touched AS (
  SELECT DISTINCT market_ticker FROM ticks WHERE yes_ask >= 0.95 AND yes_ask_size > 0
)
SELECT t.market_ticker, m.result
FROM touched t JOIN markets m ON m.market_ticker = t.market_ticker
WHERE m.result IN ('yes','no')
ORDER BY CASE WHEN m.result='no' THEN 0 ELSE 1 END, t.market_ticker
LIMIT 12
''', conn)

fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharey=True)
axes = axes.flatten()

for i, (_, row) in enumerate(sample_markets.iterrows()):
    mt = row.market_ticker
    res = row.result
    ticks_df = pd.read_sql_query(f'''
    SELECT ts, yes_ask, yes_bid, price FROM ticks
    WHERE market_ticker = '{mt}' AND yes_ask_size > 0
    ORDER BY ts
    ''', conn)
    if len(ticks_df) == 0:
        continue
    t0 = ticks_df.ts.iloc[0]
    mins = (ticks_df.ts - t0) / 60000.0
    ax = axes[i]
    ax.plot(mins, ticks_df.yes_ask, label='ask', alpha=0.7, linewidth=0.8)
    ax.plot(mins, ticks_df.yes_bid, label='bid', alpha=0.5, linewidth=0.8)
    ax.axhline(0.95, color='orange', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axhline(1.0, color='green', linestyle=':', linewidth=0.5, alpha=0.5)
    color = 'green' if res == 'yes' else 'red'
    ax.set_title(f'{mt.split("-")[-1]} ({res})', color=color, fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    if i % 4 == 0:
        ax.set_ylabel('Price')
    if i >= 8:
        ax.set_xlabel('Minutes from first tick')

axes[0].legend(fontsize=7, loc='lower right')
plt.suptitle('Price trajectories — markets that touched 95c', fontsize=13)
plt.tight_layout()
plt.show()

## Summary

| Strategy | Trades | Win% | Net P&L | Verdict |
|----------|--------|------|---------|---------|
| 95c anytime (1 entry) | 62 | 91.9% | -4.2c/share | Loses |
| 95c within 30min of close | 47 | 97.9% | +1.7c/share | Wins thin |
| 99c within 30min of close | 32 | 100% | +1.0c/share | Wins thin |
| $20/tick at 95c+ | 8512 ticks | 91.9% | -$9,925 | Loses big |
| Final set, game 8+ ($20) | 14 | 100% | +$12.38 | Wins |

**"Nothing ever happens" is conditionally true.** Near settlement in the final set, 95c markets converge to $1.
Far from close or early in the match, upsets happen ~20-33% of the time.

Caveats: 3 days of data, small samples, no fees, no slippage, 38% of markets missing final-tick convergence.